# Banking feedback: existing-data evaluation

Grouped sentiment comparison using the existing source ratings and verified classifiers.


## Reproducible source bundle


In [ ]:
import base64
import hashlib
import io
import json
import os
import subprocess
import sys
import time
import zipfile
from pathlib import Path
started=time.monotonic()
notebook=next(p for p in [Path('__notebook__.ipynb'),Path('/kaggle/working/__notebook__.ipynb')] if p.exists())
submitted=json.loads(notebook.read_text())
payload=next(c['attachments']['evaluation_assets.zip']['application/zip'] for c in submitted['cells'] if 'evaluation_assets.zip' in c.get('attachments',{}))
root=Path('/kaggle/working/evaluation_assets')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(payload))) as z:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist()), 'UNSAFE_ASSET_PATH'
    manifest=json.loads(z.read('checksums.json'))
    assert set(z.namelist())=={v['path'] for v in manifest}|{'checksums.json'}, 'ASSET_MEMBER_MISMATCH'
    for item in manifest:
        raw=z.read(item['path'])
        assert len(raw)==item['bytes'] and hashlib.sha256(raw).hexdigest()==item['sha256'], 'ASSET_HASH_MISMATCH'
    z.extractall(root)
os.environ['EVALUATION_ASSET_ROOT']=str(root)
sys.path.insert(0,str(root))
print('RESOLVED_ASSETS',str(root),len(manifest))

(root/'rating_notebook_source.json').write_text(json.dumps(submitted))


## Executable source checks


In [ ]:
from source_audit import source_audit
audits=[source_audit(p) for p in root.glob('*.py')]
directory=Path('/kaggle/working/browser_source_checks')
directory.mkdir(exist_ok=True)
cell_audits=[]
for i,cell in enumerate(submitted['cells']):
    if cell['cell_type']=='code':
        p=directory/f'cell_{i:03d}.py'
        p.write_text(''.join(cell['source']))
        cell_audits.append(source_audit(p))
(directory/'source_audits.json').write_text(json.dumps(audits,indent=2))
(directory/'notebook_code_audit.json').write_text(json.dumps(cell_audits,indent=2))
print('SOURCE_AUDITS',json.dumps(audits))
print('NOTEBOOK_SOURCE_AUDIT',json.dumps(cell_audits))


## Source records and fixed grouped split


In [ ]:
data=next(p for p in Path('/kaggle/input').rglob('labelled_reviews.parquet') if 'sarah-banking-feedback-full-dataset' in str(p))
baseline=next(p.parent for p in Path('/kaggle/input').rglob('pretraining.json') if 'sarah-banking-baseline-preparation' in str(p))
pilot=root/'pilot_unique_labels.parquet'
print('RESOLVED_DATA',str(data))
print('RESOLVED_BASELINES',str(baseline))
from rating_evaluation import prepare
frame,manifest,checks=prepare(data,pilot,20261014)
assert checks==json.loads((baseline/'pretraining.json').read_text())['data'], 'FIXED_SPLIT_MISMATCH'
print('DATA_CHECKS',json.dumps(checks))
print(frame.groupby(['split','target']).size().unstack(fill_value=0))


## Common validation comparison and reserved test


In [ ]:
from rating_evaluation import run
try:
    result=run(data,pilot,'/kaggle/working/rating_evaluation',baseline_root=baseline)
finally:
    print('TOTAL_NOTEBOOK_SECONDS',round(time.monotonic()-started,3))


## Retained evaluation


In [ ]:
print(json.dumps(result,indent=2))
assert result['status']=='complete', 'INCOMPLETE_EVALUATION'
print('EVALUATION_ARTIFACT','/kaggle/working/rating_evaluation.zip')
